# Notebook 6: Comprehensive LLM Evaluation

End-to-end LLM quality evaluation for the MedJuras RAG pipeline.

## Prerequisites
- **Notebook 1** — chunks indexed in Elasticsearch + Qdrant
- **Notebook 2** — `data/evaluation/ground_truth.json` (200 questions)
- Docker ES/Qdrant running (`local=True`)

## Evaluation tracks

### Track 1 — Offline hybrid RAG + LLM judge (`llm_judge.py`)
1. `hybrid_search` retrieves top-5 chunks (Qdrant + ES RRF)
2. Chat model generates an answer from context only
3. Judge scores **11 EU medico-legal criteria** (1–5 each)

### Track 2 — Agentic RAG evaluation (`llm_evaluation.py`)
1. Multi-iteration loop with `eu_hybrid_search` tool (same as production)
2. Judge scores **tool appropriateness**, **answer quality**, **information synthesis** (0–10)

**Three agent approaches** (same model, different temperature):
- `default` — temp **0.3**
- `conservative_agent` — temp **0.1**
- `creative_agent` — temp **0.7**

**Judge metrics (0–10):** tool appropriateness, answer quality, information synthesis, overall tool score.

**Composite score** (0–1): weighted overall + answer quality + iteration efficiency.

## Models (env-dependent)

| Role | MyGenAssist (`MYGENASSIST_API_KEY`) | 
|------|-------------------------------------|
| Answer generation | `MYGENASSIST_CHAT_MODEL` → gpt-5.1 |
| Judge | `MYGENASSIST_AUX_MODEL` → gpt-5.1 |

## Outputs (`results/`)
- `llm_evaluation.json` — combined summary
- `offline_rag_judge.json` — Track 1 (legacy name)
- `agentic_evaluation.json` — Track 2

## Cost control
Adjust `RAG_MAX_SAMPLES` and `AGENTIC_MAX_SAMPLES` below. Agentic eval is more expensive (multiple LLM + tool calls per question).
- **Ollama** (optional): llama3.2 for model comparison — see Ollama setup section below


In [1]:
import _bootstrap  
import json

from evaluation.llm_evaluator import run_full_llm_evaluation, save_llm_evaluation_results

RAG_MAX_SAMPLES = 100
AGENTIC_MAX_SAMPLES = 40
LOCAL = True
COMPARE_APPROACHES = True

results = run_full_llm_evaluation(
    rag_max_samples=RAG_MAX_SAMPLES,
    agentic_max_samples=AGENTIC_MAX_SAMPLES,
    local=LOCAL,
    compare_approaches=COMPARE_APPROACHES,
)
saved = save_llm_evaluation_results(results)

if results.get("agentic", {}).get("mode") == "compare":
    from evaluation.llm_eval_charts import best_approach_summary, comparison_from_agentic

    if saved.get("agentic_chart"):
        print("Comparison chart:", saved["agentic_chart"])
    print(best_approach_summary(comparison_from_agentic(results["agentic"])))

print("Summary:", json.dumps(results["summary"], indent=2))
print("Saved:", {k: str(v) for k, v in saved.items()})


/Users/aditya.gurung/Downloads/Medjuras.AI-Assistant-RAG-for-healthcare-research-compliance/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading ground truth:   0%|          | 0.00/38.0k [00:00<?, ?B/s]

RAG + LLM judge:   0%|          | 0/100 [00:00<?, ?query/s]

Loading ground truth:   0%|          | 0.00/38.0k [00:00<?, ?B/s]

Agentic default:   0%|          | 0/40 [00:00<?, ?query/s]

Agentic conservative_agent:   0%|          | 0/40 [00:00<?, ?query/s]

Agentic creative_agent:   0%|          | 0/40 [00:00<?, ?query/s]



Comparison chart: /Users/aditya.gurung/Downloads/Medjuras.AI-Assistant-RAG-for-healthcare-research-compliance/results/images/agentic_approach_comparison.png
Best agentic approach: Conservative (temp 0.1) | composite 0.629 | overall 6.39/10 | answer quality 6.45/10
Summary: {
  "rag_n": 100,
  "rag_mean_overall": 4.1915000000000004,
  "agentic_n": 40,
  "agentic_mean_overall": 6.3875,
  "agentic_approach": "conservative_agent",
  "agentic_tool_calls": 28
}
Saved: {'combined': '/Users/aditya.gurung/Downloads/Medjuras.AI-Assistant-RAG-for-healthcare-research-compliance/results/llm_evaluation.json', 'offline_rag_judge': '/Users/aditya.gurung/Downloads/Medjuras.AI-Assistant-RAG-for-healthcare-research-compliance/results/offline_rag_judge.json', 'agentic_evaluation': '/Users/aditya.gurung/Downloads/Medjuras.AI-Assistant-RAG-for-healthcare-research-compliance/results/agentic_evaluation.json', 'agentic_chart': '/Users/aditya.gurung/Downloads/Medjuras.AI-Assistant-RAG-for-healthcare-research-

### Ollama setup (required for MODEL_COMPARE Manual run) 

1. Install Ollama from ollama.com
2. ollama serve
3. ollama pull llama3.2
4. curl -s http://localhost:11434/api/tags


In [4]:
# import json
# import os
# import importlib

# import httpx

# # Docker: ollama_medi_rag publishes 11434 -> host uses localhost
# OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
# OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")

# os.environ["OLLAMA_BASE_URL"] = OLLAMA_BASE_URL
# os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL

# # Re-read URL if ollama_client was imported in cell 1
# from llm import ollama_client
# importlib.reload(ollama_client)


# def _check_ollama() -> None:
#     resp = httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=15.0)
#     resp.raise_for_status()
#     names = [m.get("name", "") for m in resp.json().get("models", [])]
#     if not any(OLLAMA_MODEL.split(":")[0] in n for n in names):
#         raise RuntimeError(
#             f"Model {OLLAMA_MODEL!r} not found in Ollama. "
#             f"Run: docker exec ollama_medi_rag ollama pull {OLLAMA_MODEL}"
#         )
#     print(f"Ollama OK at {OLLAMA_BASE_URL}")
#     print("Available models:", names)


# MODEL_COMPARE = True
# MODEL_MAX_SAMPLES = 40

# if MODEL_COMPARE:
#     _check_ollama()

#     from evaluation.model_comparison_eval import run_model_comparison

#     model_results = run_model_comparison(
#         max_samples=MODEL_MAX_SAMPLES,
#         local=LOCAL,
#         embedding_backend="api",  # MyGenAssist embeddings
#     )
#     print("Model comparison summary:")
#     print(json.dumps(model_results["summary"], indent=2))
#     print("JSON:", model_results["paths"]["json"])
#     print("Chart:", model_results["paths"]["chart"])